In [124]:
import os
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

In [125]:
df = pd.read_excel(r"C:\Users\yapen\Desktop\TrainDataset2025.xls")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Total rows: 400
Total columns: 121


In [126]:
df = df.drop(columns=["ID"])

In [127]:
df = df.replace(999, np.nan)

print(f'Total Number of Missing Value is {df.isna().sum().sum()}')

print(f'Total Number of Row with Missing value is {(df.isna().any(axis=1)).sum()}')

print(f'Total Number of Column with Missing value is {(df.isna().any(axis=0)).sum()}')

Total Number of Missing Value is 105
Total Number of Row with Missing value is 92
Total Number of Column with Missing value is 9


In [128]:
mask = df.isna().sum(axis=1) > 1
print(f'Row with more than 1 Missing Value: \n{df.isna().sum(axis=1)[mask]}')

# delete Rows with more than 1 Missing Value
df = df.drop(df[mask].index)

Row with more than 1 Missing Value: 
225    3
261    4
267    3
294    4
342    2
366    2
372    2
dtype: int64


In [129]:
print('After Delete Rows\n')
print(f'Total Number of Missing Value is {df.isna().sum().sum()}')

print(f'Total Number of Row with Missing value is {(df.isna().any(axis=1)).sum()}')

print(f'Total Number of Column with Missing value is {(df.isna().any(axis=0)).sum()}')

After Delete Rows

Total Number of Missing Value is 85
Total Number of Row with Missing value is 85
Total Number of Column with Missing value is 3


In [130]:
print(f"The number of rows without LNStatus = {df['LNStatus'].isna().sum()}")
print(f"The number of rows without Gene = {df['Gene'].isna().sum()}")

The number of rows without LNStatus = 1
The number of rows without Gene = 82


In [131]:
df = df.dropna(subset=['LNStatus'])
print('Rows without LNStatus are Deleted')

Rows without LNStatus are Deleted


In [133]:
gene_col = "Gene"

# ---- Drop BOTH outcomes first (to avoid leakage) ----
outcomes = ['pCR (outcome)', 'RelapseFreeSurvival (outcome)']
temp_df = df.drop(columns=outcomes)

# ---- Split rows with / without Gene ----
mask_known   = temp_df[gene_col].notna()
mask_missing = temp_df[gene_col].isna()

# ---- Use only numeric predictors excluding gene ----
feat_cols = temp_df.select_dtypes(include="number").columns.drop(gene_col)

X_train = temp_df.loc[mask_known, feat_cols]
y_train = temp_df.loc[mask_known, gene_col].astype(int)
X_missing = temp_df.loc[mask_missing, feat_cols]

# ---- Impute features ----
imputer = SimpleImputer(strategy="median")
X_train_imp   = imputer.fit_transform(X_train)
X_missing_imp = imputer.transform(X_missing)

# ---- Train classifier ----
clf = RandomForestClassifier(random_state=0)
clf.fit(X_train_imp, y_train)

# ---- Predict missing gene ----
pred_gene = clf.predict(X_missing_imp)
df.loc[mask_missing, gene_col] = pred_gene



df = df.drop(columns=['pCR (outcome)'])

In [134]:
X = df.drop(columns=['RelapseFreeSurvival (outcome)'])

y = df['RelapseFreeSurvival (outcome)']
print("Count:", len(y))
print("Min:", np.min(y))
print("Max:", np.max(y))
print("Mean:", np.mean(y))
print("Median:", np.median(y))
print("Std:", np.std(y))
print("25%:", np.percentile(y, 25))
print("75%:", np.percentile(y, 75))
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Count: 392
Min: 0.0
Max: 144.0
Mean: 56.02933673469388
Median: 55.0
Std: 27.319097906856896
25%: 38.0
75%: 73.04166666666667
Total rows: 392
Total columns: 119


In [135]:
# ===== Train-test split =====
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [136]:
X_train_df = pd.DataFrame(X_train, columns=X_train.columns)
X_test_df  = pd.DataFrame(X_test,  columns=X_test.columns)

Q1 = X_train_df.quantile(0.25)
Q3 = X_train_df.quantile(0.75)
IQR = Q3 - Q1

X_train_clip = X_train_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

X_test_clip = X_test_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

# count how many values changed (train)
train_clip_count = (X_train_df != X_train_clip).sum().sum()

# count how many values changed (test)
test_clip_count = (X_test_df != X_test_clip).sum().sum()

print("Total clipped values in train:", train_clip_count)
print("Total clipped values in test:", test_clip_count)
print("Total clipped overall:", train_clip_count + test_clip_count)


Total clipped values in train: 1425
Total clipped values in test: 564
Total clipped overall: 1989


In [137]:
# ======================================================
# 2) DETECT WHICH COLUMNS NEED SCALING
# ======================================================
unnormalized_cols = []

for col in X_train_clip.columns:
    mean = X_train_clip[col].mean()
    std  = X_train_clip[col].std()
    minv = X_train_clip[col].min()
    maxv = X_train_clip[col].max()

    is_standardized = (abs(mean) < 0.1) and (abs(std - 1) < 0.1)
    is_normalized   = (minv >= -0.1) and (maxv <= 1.1)

    if not is_standardized and not is_normalized:
        unnormalized_cols.append(col)

# column indices
cols = [X_train_clip.columns.get_loc(c) for c in unnormalized_cols]


In [138]:
# ======================================================
# 3) SCALE AFTER OUTLIER CAPPING
# ======================================================

scaler = StandardScaler()

X_train_np = X_train_clip.values
X_test_np  = X_test_clip.values

scaler.fit(X_train_np[:, cols])

X_train_std = X_train_np.copy()
X_test_std  = X_test_np.copy()

X_train_std[:, cols] = scaler.transform(X_train_np[:, cols])
X_test_std[:, cols]  = scaler.transform(X_test_np[:, cols])


X_test_std  = X_test_np.copy()

# transform only selected columns
X_train_std[:, cols] = scaler.transform(X_train_np[:, cols])

X_test_std[:, cols]  = scaler.transform(X_test_np[:, cols])


In [139]:
# ======================================================
# 4) SAVE
# ======================================================
save_dir = "regression_data_Nov_20"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "X_train.npy"), X_train_std)
np.save(os.path.join(save_dir, "X_test.npy"),  X_test_std)
np.save(os.path.join(save_dir, "y_train.npy"), y_train)
np.save(os.path.join(save_dir, "y_test.npy"),  y_test)